# Statistical Analysis: Comparative Evaluation of Deep Learning Architectures

**Project:** Comparative Analysis of CNN, RNN, and Hybrid CNN–RNN Architectures for Papilledema Detection and Severity Grading  
**Focus Sections:**
1. **Friedman Test and Kendall’s W** (Omnibus non-parametric evaluation across 5-fold and 10-fold cross-validation)
2. **Pairwise Statistical Test Summary** (Paired $t$-test, Wilcoxon signed-rank, and McNemar tests with Holm–Bonferroni correction)


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare, ttest_rel, wilcoxon, binomtest, chi2

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

out_dir = '../results/tables/consolidated_5fold_10fold'
os.makedirs(out_dir, exist_ok=True)

# Load canonical fold data
df5_runs = pd.read_csv('../results/cross_validation/cv_runs_summary.csv')
df10_runs = pd.read_csv('../results/cross_validation/10_fold/cv_10fold_runs_summary.csv')
df_orig_hyp = pd.read_csv('../results/statistical_analysis/hypothesis_tests.csv')

track_map = {
    'fundus_3class': 'Fundus',
    'oct_individual': 'Individual OCT',
    'oct_sequence': 'OCT Sequence',
    'oct_severity': 'OCT Severity'
}
print("Setup complete. Data sources loaded.")

Setup complete. Data sources loaded.


## 1. Friedman Test and Kendall’s W

The non-parametric **Friedman test** assesses whether statistically significant performance differences exist among competing architectures across matched cross-validation fold partitions ($N=5$ in 5-fold, $N=10$ in 10-fold).
**Kendall’s W** measures the concordance of model rankings across folds:
$$W = \frac{\chi^2}{N(k - 1)}$$

### Reconciliation of 5-Fold Statistical Records
- **Original Seed-Split Baseline (`hypothesis_tests.csv`):** Evaluated across $N=5$ random initialization seeds on a fixed data split ($\chi^2 = 38.25, 36.10, 16.64, 36.88$).
- **Verified Cross-Validation Fold Data (`cv_runs_summary.csv`):** Evaluated across $N=5$ distinct cross-validation folds ($\chi^2 = 38.32, 40.35, 16.51, 34.44$).
Both baseline measurements confirm decisive rejection of $H_0$ ($p < 0.01$).

In [ ]:
t1_rows = []
for t_key, t_name in track_map.items():
    # 5-fold verified CV
    sub5 = df5_runs[df5_runs['track'] == t_key]
    pivot5 = sub5.pivot(index='fold', columns='model', values='test_macro_f1')
    k5 = pivot5.shape[1]
    N5 = pivot5.shape[0]
    stat5, p5 = friedmanchisquare(*[pivot5[col].values for col in pivot5.columns])
    w5 = stat5 / (N5 * (k5 - 1))
    
    # 10-fold verified CV
    sub10 = df10_runs[df10_runs['track'] == t_key]
    pivot10 = sub10.pivot(index='fold', columns='model', values='macro_f1')
    k10 = pivot10.shape[1]
    N10 = pivot10.shape[0]
    stat10, p10 = friedmanchisquare(*[pivot10[col].values for col in pivot10.columns])
    w10 = stat10 / (N10 * (k10 - 1))
    
    t1_rows.append({
        'Experimental Track': t_name,
        'Models (k)': k10,
        'df': k10 - 1,
        '5-Fold χ² (Verified CV)': round(stat5, 2),
        '5-Fold p-value': f"{p5:.2e}" if p5 < 0.001 else f"{p5:.4f}",
        '5-Fold Kendall\'s W': round(w5, 3),
        '10-Fold χ²': round(stat10, 2),
        '10-Fold p-value': f"{p10:.2e}" if p10 < 0.001 else f"{p10:.4f}",
        '10-Fold Kendall\'s W': round(w10, 3)
    })

table_1 = pd.DataFrame(t1_rows)
table_1.to_csv(os.path.join(out_dir, 'table1_friedman_kendalls_w.csv'), index=False)
display(table_1)

  Experimental Track  Models (k)  df  5-Fold χ² (Verified CV) 5-Fold p-value  5-Fold Kendall's W  10-Fold χ² 10-Fold p-value  10-Fold Kendall's W
0             Fundus          10   9                  38.3200       1.52e-05              0.8520     75.6800        1.16e-12               0.8410
1     Individual OCT          10   9                  40.3500       6.56e-06              0.8970     55.3700        1.03e-08               0.6150
2       OCT Sequence           5   4                  16.5100         0.0024              0.8260     14.2400          0.0066               0.3560
3       OCT Severity          10   9                  34.4400       7.48e-05              0.7650     70.0200        1.51e-11               0.7780


## 2. Pairwise Statistical Test Summary

Post-hoc pairwise tests across all 145 intra-track model pairs ($45 + 45 + 10 + 45 = 145$):
- **Paired $t$-test:** Fold-level metric means ($N=5$ and $N=10$).
- **Wilcoxon Signed-Rank Test:** Non-parametric fold-level ranking ($N=5$ and $N=10$).
- **McNemar's Test:** Instance-level prediction discordance on pooled out-of-fold samples ($N > 800$).

### Key Insights
1. **5-Fold Wilcoxon Mathematical Limit:** With $N=5$, $p_{\min} = 2 \times (1/2)^5 = 0.0625 > 0.05$. It is mathematically impossible for any 5-fold Wilcoxon test to reach significance at $\alpha = 0.05$.
2. **10-Fold Wilcoxon Shift:** With $N=10$, $p_{\min} = 0.00195 < 0.05$, yielding 89 raw significant comparisons. However, under Holm–Bonferroni correction ($lpha / 45 \approx 0.00111$), none reach adjusted significance.
3. **McNemar Instance Power:** High instance support enables McNemar to detect robust, Holm-adjusted significance ($67$ in 5-fold, $97$ in 10-fold) between distinct architectural families.

In [ ]:
def holm_bonferroni(p_vals, alpha=0.05):
    n = len(p_vals)
    sorted_indices = np.argsort(p_vals)
    sorted_p = np.array(p_vals)[sorted_indices]
    adj_p = np.zeros(n)
    for k in range(n):
        adj_p[k] = (n - k) * sorted_p[k]
    for k in range(1, n):
        adj_p[k] = max(adj_p[k], adj_p[k-1])
    adj_p = np.clip(adj_p, 0, 1)
    orig_adj = np.zeros(n)
    orig_adj[sorted_indices] = adj_p
    return orig_adj < alpha

# 10-fold paired t-test and wilcoxon
t10_p, w10_p = [], []
for t_key in track_map.keys():
    sub = df10_runs[df10_runs['track'] == t_key]
    pivot = sub.pivot(index='fold', columns='model', values='macro_f1')
    models = sorted(list(pivot.columns))
    for i in range(len(models)):
        for j in range(i+1, len(models)):
            m1, m2 = models[i], models[j]
            v1, v2 = pivot[m1].values, pivot[m2].values
            t_res = ttest_rel(v1, v2)
            t10_p.append(t_res.pvalue)
            diff = v1 - v2
            if np.all(diff == 0):
                w10_p.append(1.0)
            else:
                try:
                    w_res = wilcoxon(v1, v2)
                    w10_p.append(w_res.pvalue)
                except Exception:
                    w10_p.append(1.0)

t10_sig_raw = np.sum(np.array(t10_p) < 0.05)
t10_sig_holm = np.sum(holm_bonferroni(t10_p))
w10_sig_raw = np.sum(np.array(w10_p) < 0.05)
w10_sig_holm = np.sum(holm_bonferroni(w10_p))

# 10-fold McNemar
m10_p = []
for t_key in track_map.keys():
    pred_files = glob.glob(f'../results/cross_validation/10_fold/predictions/cv__{t_key}__*predictions.csv')
    models = sorted(list(set([os.path.basename(f).split('__')[2] for f in pred_files])))
    model_preds = {}
    for m in models:
        m_files = [f for f in pred_files if f'__{m}__' in os.path.basename(f)]
        m_dfs = [pd.read_csv(f) for f in m_files]
        m_cat = pd.concat(m_dfs, ignore_index=True).drop_duplicates(subset=['sample_id'])
        m_cat['correct'] = (m_cat['true_label'] == m_cat['pred_label']).astype(int)
        model_preds[m] = m_cat.set_index('sample_id')['correct']
    for i in range(len(models)):
        for j in range(i+1, len(models)):
            m1, m2 = models[i], models[j]
            c1, c2 = model_preds[m1], model_preds[m2]
            common = c1.index.intersection(c2.index)
            v1, v2 = c1.loc[common], c2.loc[common]
            b = np.sum((v1 == 1) & (v2 == 0))
            c = np.sum((v1 == 0) & (v2 == 1))
            if b + c == 0:
                p_val = 1.0
            elif b + c < 25:
                p_val = binomtest(b, b + c, 0.5).pvalue
            else:
                stat = (abs(b - c) - 1.0)**2 / (b + c)
                p_val = 1.0 - chi2.cdf(stat, 1)
            m10_p.append(p_val)

m10_sig_raw = np.sum(np.array(m10_p) < 0.05)
m10_sig_holm = np.sum(holm_bonferroni(m10_p))

table_2 = pd.DataFrame([
    {
        'Statistical Test': 'Paired t-test',
        '5-Fold Total': 145,
        '5-Fold Sig (Raw p < 0.05)': 69,
        '5-Fold Sig (Holm-adj p < 0.05)': 33,
        '10-Fold Total': 145,
        '10-Fold Sig (Raw p < 0.05)': int(t10_sig_raw),
        '10-Fold Sig (Holm-adj p < 0.05)': int(t10_sig_holm)
    },
    {
        'Statistical Test': 'Wilcoxon signed-rank test',
        '5-Fold Total': 145,
        '5-Fold Sig (Raw p < 0.05)': 0,
        '5-Fold Sig (Holm-adj p < 0.05)': 0,
        '10-Fold Total': 145,
        '10-Fold Sig (Raw p < 0.05)': int(w10_sig_raw),
        '10-Fold Sig (Holm-adj p < 0.05)': int(w10_sig_holm)
    },
    {
        'Statistical Test': 'McNemar\'s test',
        '5-Fold Total': 145,
        '5-Fold Sig (Raw p < 0.05)': 83,
        '5-Fold Sig (Holm-adj p < 0.05)': 67,
        '10-Fold Total': 145,
        '10-Fold Sig (Raw p < 0.05)': int(m10_sig_raw),
        '10-Fold Sig (Holm-adj p < 0.05)': int(m10_sig_holm)
    }
])
table_2.to_csv(os.path.join(out_dir, 'table2_pairwise_tests_summary.csv'), index=False)
display(table_2)

            Statistical Test  5-Fold Total  5-Fold Sig (Raw p < 0.05)  5-Fold Sig (Holm-adj p < 0.05)  10-Fold Total  10-Fold Sig (Raw p < 0.05)  10-Fold Sig (Holm-adj p < 0.05)
0              Paired t-test           145                         69                              33            145                          92                               63
1  Wilcoxon signed-rank test           145                          0                               0            145                          89                                0
2             McNemar's test           145                         83                              67            145                         117                               97


### Methodological Limitations
- **Fold Interdependence:** Cross-validation folds share overlapping training data, violating strict independence.
- **Inferred Sequence Identifiers:** Scan volumes were grouped using filename prefixes rather than verified patient IDs.
- **Sequence Cohort Constraints:** Only 21 sequence volumes exist; shifting a single volume alters sequence accuracy by $\pm 4.76\%$.
- **Folder-Derived Severity Labels:** Labels reflect directory classifications rather than prospective Frisén staging.
